
# Customer Churn Prediction – End to End Data Science Case Study

**Author:** Kehinde Damilola Akindele  
**Use case:** Subscription business (for example telecom, SaaS, or streaming)  

**Objective:**  
Build and evaluate machine learning models to predict customer churn, understand key drivers of churn, and translate results into clear business recommendations.  

This notebook is structured as a portfolio-quality project and includes:

1. Problem definition and data understanding  
2. Exploratory data analysis (EDA)  
3. Data preprocessing and feature engineering  
4. Model training and evaluation (Logistic Regression and Random Forest)  
5. Feature importance and insights  
6. Business recommendations and next steps  
7. Export of a consolidated PDF report (methodology, key results, recommendations)  



## 1. Problem definition and data

A subscription-based company wants to **reduce customer churn**.  
Management asks:

- Can we **predict which customers are likely to churn** in the near future?  
- What are the **main drivers** of churn?  
- How can we **target retention actions** more effectively?  

### Dataset

We use a public **Telco Customer Churn** dataset that contains:

- Customer demographics (gender, senior citizen, partner, dependents)  
- Contract information (contract type, payment method, tenure)  
- Charges (monthly charges, total charges)  
- Target variable `Churn` (Yes / No)  

> ⚠️ **Note:** This notebook assumes a CSV file similar to the Kaggle “Telco Customer Churn” dataset.  
> Place the file in the same folder as this notebook and set the file name below (for example `WA_Fn-UseC_-Telco-Customer-Churn.csv`).  


In [ ]:

# 1.1 Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import seaborn as sns

# For exporting a standalone PDF report at the end
from datetime import datetime

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas
    REPORTLAB_AVAILABLE = True
except ImportError:
    REPORTLAB_AVAILABLE = False

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

print("ReportLab available for PDF export:", REPORTLAB_AVAILABLE)


In [ ]:

# 1.2 Load the dataset
#
# Set this to the correct file name for your environment.
# Example for Kaggle telco churn data:
# data_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

data_path = "telco_churn.csv"  # change this if needed

df = pd.read_csv(data_path)

print("Shape of raw data:", df.shape)
df.head()


In [ ]:

# 1.3 Basic info and data types
#
# This helps us understand which columns are numeric vs categorical,
# and if we have missing values or unexpected data types.

df.info()


In [ ]:

# 1.4 Basic summary statistics for numeric columns

df.describe()



## 2. Exploratory data analysis (EDA)

In this section we:

- Inspect the distribution of the churn target  
- Explore relationships between churn and key features (tenure, charges, contract type)  
- Look at correlations between numeric variables  


In [ ]:

# 2.1 Target distribution
#
# Understanding class balance is important for model choice and evaluation.

churn_counts = df["Churn"].value_counts()
churn_ratio = df["Churn"].value_counts(normalize=True)

print("Churn counts:")
print(churn_counts)
print("\nChurn ratio:")
print(churn_ratio)

plt.figure(figsize=(4, 4))
sns.countplot(data=df, x="Churn")
plt.title("Churn distribution")
plt.show()


In [ ]:

# 2.2 Relationship between churn and tenure

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure by churn")
plt.show()


In [ ]:

# 2.3 Relationship between churn and monthly charges

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly charges by churn")
plt.show()


In [ ]:

# 2.4 Churn rate by contract type (if the column exists)

if "Contract" in df.columns:
    contract_churn = (
        df.groupby("Contract")["Churn"]
        .value_counts(normalize=True)
        .rename("churn_rate")
        .reset_index()
    )

    plt.figure(figsize=(6, 4))
    sns.barplot(
        data=contract_churn[contract_churn["Churn"] == "Yes"],
        x="Contract",
        y="churn_rate",
    )
    plt.title("Churn rate by contract type")
    plt.ylabel("Churn rate")
    plt.xticks(rotation=20)
    plt.show()
else:
    print("Column 'Contract' not found, skipping contract churn plot.")


In [ ]:

# 2.5 Correlation between numeric features

numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    plt.figure(figsize=(6, 5))
    corr = df[numeric_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues")
    plt.title("Correlation between numeric features")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns found for correlation matrix.")



### EDA observations (update based on your results)

After inspecting the plots, you can note observations like:

- The dataset is **moderately imbalanced**, with fewer churned customers than non-churned.  
- Customers with **shorter tenure** tend to churn more often than long-term customers.  
- Customers with **higher monthly charges** show a higher churn rate, suggesting price sensitivity.  
- Month-to-month contracts often have higher churn than long-term contracts (if present in your dataset).  
- Numeric correlations are moderate; tenure and total charges tend to be correlated.  



## 3. Data preprocessing and feature engineering

Steps:

1. Handle missing values and clean data types (especially `TotalCharges`)  
2. Encode the target variable `Churn` as 0/1  
3. Separate features and target  
4. Build preprocessing pipelines:  
   - Standardize numeric features  
   - One hot encode categorical features  
5. Split the data into train and test sets  


In [ ]:

# 3.1 Basic cleaning
#
# Some versions of the Telco dataset have spaces for missing values,
# especially in `TotalCharges`. We replace blank strings with NaN,
# then drop or impute as needed for this demo.

df = df.replace(" ", np.nan)

# Convert TotalCharges to numeric if needed
if "TotalCharges" in df.columns and df["TotalCharges"].dtype == "object":
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop rows with missing TotalCharges for simplicity
if "TotalCharges" in df.columns:
    before_rows = df.shape[0]
    df = df.dropna(subset=["TotalCharges"])
    after_rows = df.shape[0]
    print(f"Dropped {before_rows - after_rows} rows due to missing TotalCharges.")

print("Shape after cleaning:", df.shape)
df.head()


In [ ]:

# 3.2 Encode target variable
#
# Map: Yes -> 1 (churn), No -> 0 (stay)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(df["Churn"].value_counts())


In [ ]:

# 3.3 Separate features and target
#
# We drop `customerID` because it is an identifier, not a predictive feature.

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


In [ ]:

# 3.4 Define preprocessing pipelines

numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())]
)

categorical_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 3.5 Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)



## 4. Model training and evaluation

We train and compare two models:

1. **Logistic Regression** – simple, interpretable baseline  
2. **Random Forest** – non-linear, often stronger performance  

We evaluate using:

- Accuracy  
- Precision, recall, F1-score  
- ROC AUC  
- Confusion matrix  


In [ ]:

# Helper function to evaluate classifiers

def evaluate_model(y_true, y_pred, y_proba, name: str):
    metrics = {}
    metrics["accuracy"] = accuracy_score(y_true, y_pred)
    metrics["precision"] = precision_score(y_true, y_pred)
    metrics["recall"] = recall_score(y_true, y_pred)
    metrics["f1"] = f1_score(y_true, y_pred)
    metrics["roc_auc"] = roc_auc_score(y_true, y_proba)

    print(f"=== {name} ===")
    print("Accuracy :", round(metrics['accuracy'], 3))
    print("Precision:", round(metrics['precision'], 3))
    print("Recall   :", round(metrics['recall'], 3))
    print("F1-score :", round(metrics['f1'], 3))
    print("ROC AUC  :", round(metrics['roc_auc'], 3))
    print()
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion matrix – {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return metrics


In [ ]:

# 4.1 Logistic Regression

log_reg_clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

log_reg_clf.fit(X_train, y_train)

y_pred_lr = log_reg_clf.predict(X_test)
y_proba_lr = log_reg_clf.predict_proba(X_test)[:, 1]

metrics_lr = evaluate_model(y_test, y_pred_lr, y_proba_lr, "Logistic Regression")


In [ ]:

# 4.2 Random Forest

rf_clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
            ),
        ),
    ]
)

rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)
y_proba_rf = rf_clf.predict_proba(X_test)[:, 1]

metrics_rf = evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")


In [ ]:

# 4.3 Compare models and choose a "best model"

models_summary = pd.DataFrame(
    [
        {"model": "Logistic Regression", **metrics_lr},
        {"model": "Random Forest", **metrics_rf},
    ]
)

models_summary


In [ ]:

# Choose best model based on ROC AUC (you can change the criterion)

if metrics_rf["roc_auc"] >= metrics_lr["roc_auc"]:
    best_model_name = "Random Forest"
    best_model = rf_clf
    best_metrics = metrics_rf
else:
    best_model_name = "Logistic Regression"
    best_model = log_reg_clf
    best_metrics = metrics_lr

print("Best model based on ROC AUC:", best_model_name)
print("Best model metrics:", {k: round(v, 3) for k, v in best_metrics.items()})



## 5. Feature importance and insights

For the Random Forest model we can inspect **feature importances** to understand which variables drive churn predictions.  

This helps translate model results into business language:

- Which characteristics are associated with higher churn risk?  
- What aspects of the customer experience might we need to improve?  


In [ ]:

# 5.1 Extract feature names after preprocessing and compute importances
#
# If the best model is Random Forest, use its feature importances.
# If Logistic Regression is best, you could adapt this section to use coefficients instead.

top_features_df = None

if best_model_name == "Random Forest":
    # Get fitted OneHotEncoder
    ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["encoder"]
    encoded_cat_features = ohe.get_feature_names_out(categorical_features)
    all_features = np.concatenate([numeric_features, encoded_cat_features])

    rf_model = best_model.named_steps["model"]
    importances = rf_model.feature_importances_

    feat_imp = pd.DataFrame(
        {"feature": all_features, "importance": importances}
    ).sort_values("importance", ascending=False)

    top_n = 15
    top_features_df = feat_imp.head(top_n).reset_index(drop=True)

    display(top_features_df)

    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=top_features_df,
        x="importance",
        y="feature",
    )
    plt.title("Top feature importances for churn prediction (Random Forest)")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("Best model is Logistic Regression. For coefficients-based interpretation, adapt this section.")



### Interpretation (adjust based on your results)

Typically, we observe patterns such as:

- **Tenure** among the most important features – shorter tenure often correlates with higher churn.  
- **MonthlyCharges** and **TotalCharges** influencing churn – customers paying more may be more likely to leave if they do not perceive enough value.  
- **Contract type** – month-to-month contracts often see higher churn than one- or two-year contracts.  
- Service-related features (for example internet service type, additional services) can also differentiate stable customers from those at risk.  



## 6. Business recommendations and next steps

Based on modeling and insights, here are example recommendations you can refine for your own case:

### 6.1 Targeted retention actions

- Use the **best-performing model** to score active customers monthly.  
- Focus retention campaigns on customers in the **top churn-risk deciles** (for example top 20–30% by predicted probability).  
- Prioritize high-value customers (for example high monthly charges) with high churn risk.  

### 6.2 Product and pricing strategy

- For segments with **short tenure and high charges**, review onboarding, communications, and perceived value.  
- Consider targeted discounts or enhanced service bundles to reduce churn in those groups.  
- For customers on **month-to-month contracts**, offer incentives to move them to longer-term contracts with historically lower churn.  

### 6.3 Model and data improvements

- Tune model hyperparameters on a validation set or via cross-validation for further performance gains.  
- Add **behavioural features** such as product usage, customer support tickets, and marketing engagement.  
- Set up a **monitoring pipeline** to track model performance over time and retrain as needed.  



## 7. Export consolidated PDF report

In this final step, we generate a **standalone PDF report** that summarizes:

- Problem statement and dataset  
- Methodology and models used  
- Key performance metrics for the best model  
- Top features driving churn  
- Business recommendations  

This allows you to attach a single PDF to job applications (for example Upwork proposals) or share with stakeholders.

> 🔧 **Implementation detail:**  
> This section uses the `reportlab` library.  
> If you do not have it installed, run `pip install reportlab` in your environment and re-run this section.


In [ ]:

# 7.1 Build a textual report and export to PDF (if ReportLab is available)

def build_text_report():
    lines = []

    lines.append("Customer Churn Prediction – Data Science Case Study")
    lines.append("Author: Kehinde Damilola Akindele")
    lines.append("Generated on: " + datetime.now().strftime("%Y-%m-%d %H:%M"))
    lines.append("")
    lines.append("1. Problem statement")
    lines.append(
        "A subscription-based company wants to reduce customer churn by predicting which customers are at risk "
        "and understanding the drivers behind churn."
    )
    lines.append("")
    lines.append("2. Dataset")
    lines.append(
        "Telco customer churn dataset with customer demographics, contract information, charges, and a binary "
        "churn flag (Yes/No)."
    )
    lines.append("")
    lines.append("3. Methodology (high level)")
    lines.append(
        "- Performed exploratory data analysis (EDA) to understand churn distribution and key relationships "
        "(tenure, charges, contract type)."
    )
    lines.append(
        "- Cleaned and preprocessed data, encoded categorical variables, and standardized numeric variables."
    )
    lines.append(
        "- Trained two supervised learning models: Logistic Regression and Random Forest."
    )
    lines.append(
        "- Evaluated models using accuracy, precision, recall, F1-score, and ROC AUC on a held-out test set."
    )
    lines.append(
        "- Selected the best model based on ROC AUC and inspected feature importances to extract insights."
    )
    lines.append("")

    lines.append("4. Best model performance")
    lines.append(f"- Best model: {best_model_name}")
    for metric_name, metric_value in best_metrics.items():
        lines.append(f"- {metric_name.capitalize()}: {metric_value:.3f}")
    lines.append("")

    if top_features_df is not None:
        lines.append("5. Top features driving churn (Random Forest importances)")
        for _, row in top_features_df.iterrows():
            lines.append(
                f"- {row['feature']}: importance {row['importance']:.4f}"
            )
        lines.append("")
    else:
        lines.append(
            "5. Feature importance: best model is Logistic Regression; inspect coefficients in the notebook for details."
        )
        lines.append("")

    lines.append("6. Business recommendations (summary)")
    lines.append(
        "- Use the best model to score customers regularly and focus interventions on those with the highest predicted churn risk."
    )
    lines.append(
        "- Pay particular attention to high-value customers (for example high monthly charges) with high churn scores."
    )
    lines.append(
        "- For short-tenure customers and those on flexible contracts, review onboarding, communications, and value proposition."
    )
    lines.append(
        "- Consider model and data improvements such as hyperparameter tuning and adding behavioural features."
    )
    lines.append("")
    lines.append("End of report.")
    return lines


def export_pdf_report(file_name: str = "churn_case_study_report.pdf"):
    if not REPORTLAB_AVAILABLE:
        print(
            "ReportLab is not installed. Install it with `pip install reportlab` and re-run this cell."
        )
        return

    lines = build_text_report()

    c = canvas.Canvas(file_name, pagesize=A4)
    width, height = A4

    # Simple line-by-line text writer with automatic page breaks
    x_margin = 40
    y_margin = 40
    max_width = width - 2 * x_margin
    y = height - y_margin

    c.setFont("Helvetica", 11)

    for line in lines:
        # Basic wrapping: split long lines into chunks
        while len(line) * 5 > max_width:  # crude wrap estimate
            # find split position
            split_pos = max_width // 5
            part = line[: int(split_pos)]
            c.drawString(x_margin, y, part)
            y -= 14
            line = line[int(split_pos) :]
            if y < y_margin:
                c.showPage()
                c.setFont("Helvetica", 11)
                y = height - y_margin

        c.drawString(x_margin, y, line)
        y -= 14

        if y < y_margin:
            c.showPage()
            c.setFont("Helvetica", 11)
            y = height - y_margin

    c.save()
    print(f"PDF report exported to: {file_name}")


# Run the export
export_pdf_report("customer_churn_case_study_report.pdf")
